In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from models import DeeperNN, SmallerNN
from datasets import UNSW_NB15_Dataset
from losses import SqrHingeLoss

import copy

### Data import and cleaning 

In [ ]:
data_tr = pd.read_csv('../datasets/UNSW_NB15_training-set.csv', delimiter=',')
data_tst = pd.read_csv('../datasets/UNSW_NB15_testing-set.csv', delimiter=',')

In [ ]:



X_train = torch.tensor(samples.values, dtype=torch.float32)
Y_train = torch.tensor(labels.values, dtype=torch.long)

In [ ]:
def train_eval_loop(train_dataloader: DataLoader, model: nn.Module, loss_fn, lr, epochs, patience, X_val=None, Y_val=None,fold_idx=None, dummy_size=None, bin=False):
    size = len(train_dataloader.dataset)

    optimizer = torch.optim.Adam(model.parameters(), lr)
    if dummy_size == None:
        scheduler = ReduceLROnPlateau(optimizer=optimizer, patience=patience, verbose=True, min_lr=1e-6)

    best_acc = -np.inf
    best_weights = None
    
    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")

        model.train()
        loss_fn.train()
        
        for batch, (X, Y) in enumerate(train_dataloader):
            if dummy_size != None and batch >= dummy_size:
                break

            if isinstance(loss_fn, SqrHingeLoss):
                target = Y.unsqueeze(1)
                target_onehot = torch.Tensor(target.size(0), 2)
                target_onehot.fill_(-1)
                target_onehot.scatter_(1, target, 1)
                target = target.squeeze()
                target_var = target_onehot
            else:
                target_var = Y
            
            # forward pass
            pred = model(X)
            loss = loss_fn(pred, target_var)
            # backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if(bin and hasattr(model, 'clip_weights')):
                model.clip_weights(-1, 1)

            # training stats every x batches
            if(batch % 50 == 0):
                loss, current = loss.item(), batch * train_dataloader.batch_size + len(X)
                acc = (pred.argmax(1).round() == Y).float().mean()
                print(f"loss: {loss:>7f} acc: {acc*100}% [{current:>5d}/{size:>5d}]")

        if(X_val != None and Y_val != None):
            model.eval()

            if isinstance(loss_fn, SqrHingeLoss):
                target = Y_val.unsqueeze(1)
                target_onehot = torch.Tensor(target.size(0), 2)
                target_onehot.fill_(-1)
                target_onehot.scatter_(1, target, 1)
                target = target.squeeze()
                target_var = target_onehot
            else:
                target_var = Y_val

            pred = model(X_val)
            val_loss = loss_fn(pred, target_var)
            acc = (pred.argmax(1).round() == Y_val).float().mean()
            print(f'evaluation fold {fold_idx} acc: {acc} ')
            if acc > best_acc:
                best_acc = acc
                best_weights = copy.deepcopy(model.state_dict())
        
        # lr decay
        if dummy_size == None:
            scheduler.step(val_loss)  
    
    return best_weights, best_acc

#### Hyper parameter definition

In [ ]:
batch_size = 1024
epochs = 20
lr = 1e-1
kfold_splits = 3
patience = 1
hidden_layers = [45, 23, 19]
hidden_layers_small = [45, 45]
input_neuron_size = 59
criterion = SqrHingeLoss()

### Train and validation phase

In [ ]:
# sss = StratifiedShuffleSplit(n_splits=kfold_splits, random_state=0, test_size=0.1)
# best_model = None
# best_acc = -np.inf
# kfold_idx = 0
# last_act = None # softmax default

# for train_ids, val_ids in sss.split(X_train, Y_train):
#     if isinstance(criterion, SqrHingeLoss): last_act = 'tanh'
#     model = DeeperNN(hidden_layers=hidden_layers, input_size=input_neuron_size, last_act=last_act)
#     train_dataset = UNSW_NB15_Dataset(X_train[train_ids], Y_train[train_ids])
#     train_dataloader = DataLoader(train_dataset, batch_size=batch_size)
#     X_val, Y_val = X_train[val_ids], Y_train[val_ids]
#     model_dict, acc = train_eval_loop(train_dataloader, model, criterion, lr, epochs, patience, X_val, Y_val, kfold_idx)
    
#     if(acc > best_acc):
#         best_acc=acc
#         best_model=model_dict
    
#     print(f'\n*  FULL MODEL: fold no {kfold_idx} ended - best acc: {best_acc:.3f}  *\n')
#     kfold_idx += 1

# torch.save(best_model, f'models/bnn_full_acc_{best_acc:.3f}.pth')

In [ ]:
sss = StratifiedShuffleSplit(n_splits=kfold_splits, random_state=0, test_size=0.1)
best_model = None
best_acc = -np.inf
kfold_idx = 0
last_act = None # softmax default

for train_ids, val_ids in sss.split(X_train, Y_train):
    if isinstance(criterion, SqrHingeLoss): last_act = 'tanh'
    model = SmallerNN(hidden_layers=hidden_layers_small, input_size=input_neuron_size)
    train_dataset = UNSW_NB15_Dataset(X_train[train_ids], Y_train[train_ids])
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size)
    X_val, Y_val = X_train[val_ids], Y_train[val_ids]
    model_dict, acc = train_eval_loop(train_dataloader, model, criterion, lr, epochs, patience, X_val, Y_val, kfold_idx, bin=True)
    
    if(acc > best_acc):
        best_acc=acc
        best_model=model_dict

    print(f'\n* QUANT MODEL: fold no {kfold_idx} ended - best acc: {best_acc:.3f}  *\n')
    kfold_idx += 1

torch.save(best_model, f'models/bnn_quant_acc_{best_acc:.3f}.pth')

kfold_idx = 0